In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ============================================================
# STAGE 1: DATA LOADING
# ============================================================
print("="*60)
print("STAGE 1: LOADING DATA")
print("="*60)

df = pd.read_csv('/kaggle/input/ipl-dataset2008-2025/IPL.csv')

print(f"\nOriginal shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique matches: {df['match_id'].nunique()}")
print(f"Unique batters: {df['batter'].nunique()}")
print(f"Unique bowlers: {df['bowler'].nunique()}")


STAGE 1: LOADING DATA


/tmp/ipykernel_47/2140331996.py:14: DtypeWarning: Columns (28,29,30,31,43,46,47,48,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/kaggle/input/ipl-dataset2008-2025/IPL.csv')



Original shape: (278205, 64)
Date range: 2008-04-18 to 2025-06-03
Unique matches: 1169
Unique batters: 703
Unique bowlers: 550


In [2]:
# ============================================================
# STAGE 2: DATA CLEANING
# ============================================================
print("\n" + "="*60)
print("STAGE 2: DATA CLEANING")
print("="*60)

columns_to_drop = [
    'Unnamed: 0', 'review_batter', 'team_reviewed', 'review_decision',
    'umpire', 'umpires_call', 'ball_no', 'runs_not_boundary',
    'non_striker_pos', 'fielders', 'superover_winner'
]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print(f"\nAfter dropping unnecessary columns: {df.shape}")

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date', 'match_id', 'innings', 'over', 'ball']).reset_index(drop=True)

print(f"\nDate range after conversion: {df['date'].min()} to {df['date'].max()}")

print("\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0])

critical_columns = ['match_id', 'date', 'batter', 'bowler', 'batting_team', 'bowling_team']
df = df.dropna(subset=critical_columns)

print(f"\nAfter handling missing values: {df.shape}")

df = df[df['event_name'] == 'Indian Premier League'].copy()
print(f"After filtering to IPL only: {df.shape}")



STAGE 2: DATA CLEANING

After dropping unnecessary columns: (278205, 53)

Date range after conversion: 2008-04-18 00:00:00 to 2025-06-03 00:00:00

Missing values:
extra_type     263072
wicket_kind    264382
player_out     264382
runs_target    144302
win_outcome      4702
result_type    273503
method         274315
new_batter     264884
next_batter    264884
dtype: int64

After handling missing values: (278205, 53)
After filtering to IPL only: (278205, 53)


In [3]:
# ============================================================
# STAGE 3: FEATURE ENGINEERING - BASIC FEATURES
# ============================================================
print("\n" + "="*60)
print("STAGE 3: BASIC FEATURE ENGINEERING")
print("="*60)

df['is_wicket'] = df['wicket_kind'].notna().astype(int)

df['is_boundary'] = df['runs_batter'].isin([4, 6]).astype(int)

df['ball_id'] = df.groupby('match_id').cumcount() + 1

print(f"\nTotal wickets in dataset: {df['is_wicket'].sum():,}")
print(f"Total boundaries: {df['is_boundary'].sum():,}")



STAGE 3: BASIC FEATURE ENGINEERING

Total wickets in dataset: 13,823
Total boundaries: 46,466


In [4]:
# ============================================================
# STAGE 4: AGGREGATE TO PLAYER-MATCH LEVEL - BATSMEN
# ============================================================
print("\n" + "="*60)
print("STAGE 4: BATSMAN MATCH-LEVEL AGGREGATION")
print("="*60)

batsman_stats = df.groupby(
    ['match_id', 'date', 'batter', 'batting_team', 'bowling_team', 'venue', 'city', 'season']
).agg({
    'runs_batter': 'sum',
    'valid_ball': 'sum',
    'is_boundary': 'sum',
    'is_wicket': 'sum',
    'innings': 'first'
}).reset_index()

batsman_stats.columns = [
    'match_id', 'date', 'player', 'team', 'opponent', 'venue', 'city', 
    'season', 'runs', 'balls_faced', 'boundaries', 'got_out', 'innings'
]

print(f"\nPlayers with 0 balls faced: {(batsman_stats['balls_faced'] == 0).sum()}")
batsman_stats = batsman_stats[batsman_stats['balls_faced'] > 0].copy()
print(f"After removing 0 balls faced: {batsman_stats.shape}")

batsman_stats['strike_rate'] = (
    batsman_stats['runs'] / batsman_stats['balls_faced'] * 100
)

batsman_stats['batting_first'] = (batsman_stats['innings'] == 1).astype(int)

print(f"\nBatsman stats shape: {batsman_stats.shape}")
print(f"Unique batsmen: {batsman_stats['player'].nunique()}")
print(f"Average runs per innings: {batsman_stats['runs'].mean():.2f}")
print(f"Average balls faced: {batsman_stats['balls_faced'].mean():.2f}")

print("\nSample batsman stats:")
print(batsman_stats.head(10))

print("\n" + "="*60)
print("DATA QUALITY VERIFICATION - BATSMEN")
print("="*60)

duplicates = batsman_stats.groupby(['match_id', 'player']).size()
print(f"Duplicate player-match entries: {(duplicates > 1).sum()}")

if 'V Kohli' in batsman_stats['player'].values:
    print("\nSample verification for V Kohli:")
    kohli_sample = batsman_stats[batsman_stats['player'] == 'V Kohli'].head()
    print(kohli_sample[['date', 'runs', 'balls_faced', 'strike_rate', 'boundaries']])



STAGE 4: BATSMAN MATCH-LEVEL AGGREGATION

Players with 0 balls faced: 8
After removing 0 balls faced: (17640, 13)

Batsman stats shape: (17640, 15)
Unique batsmen: 703
Average runs per innings: 20.15
Average balls faced: 15.19

Sample batsman stats:
   match_id       date           player                         team  \
0    335982 2008-04-18        AA Noffke  Royal Challengers Bangalore   
1    335982 2008-04-18          B Akhil  Royal Challengers Bangalore   
2    335982 2008-04-18      BB McCullum        Kolkata Knight Riders   
3    335982 2008-04-18         CL White  Royal Challengers Bangalore   
4    335982 2008-04-18        DJ Hussey        Kolkata Knight Riders   
5    335982 2008-04-18        JH Kallis  Royal Challengers Bangalore   
6    335982 2008-04-18       MV Boucher  Royal Challengers Bangalore   
7    335982 2008-04-18  Mohammad Hafeez        Kolkata Knight Riders   
8    335982 2008-04-18          P Kumar  Royal Challengers Bangalore   
9    335982 2008-04-18       

In [5]:
# ============================================================
# STAGE 5: AGGREGATE TO PLAYER-MATCH LEVEL - BOWLERS
# ============================================================
print("\n" + "="*60)
print("STAGE 5: BOWLER MATCH-LEVEL AGGREGATION")
print("="*60)

bowler_stats = df.groupby(
    ['match_id', 'date', 'bowler', 'bowling_team', 'batting_team', 'venue', 'city', 'season']
).agg({
    'runs_bowler': 'sum',
    'valid_ball': 'sum',
    'is_wicket': 'sum',
    'innings': 'first'
}).reset_index()

bowler_stats.columns = [
    'match_id', 'date', 'player', 'team', 'opponent', 'venue', 'city',
    'season', 'runs_conceded', 'balls_bowled', 'wickets', 'innings'
]

print(f"\nBowlers with 0 balls bowled: {(bowler_stats['balls_bowled'] == 0).sum()}")
bowler_stats['economy'] = (
    bowler_stats['runs_conceded'] / bowler_stats['balls_bowled'].replace(0, np.nan) * 6
).fillna(0)

bowler_stats['overs_bowled'] = bowler_stats['balls_bowled'] / 6

bowler_stats['bowling_first'] = (bowler_stats['innings'] == 1).astype(int)

print(f"\nBowler stats shape: {bowler_stats.shape}")
print(f"Unique bowlers: {bowler_stats['player'].nunique()}")
print(f"Average wickets per match: {bowler_stats['wickets'].mean():.2f}")
print(f"Average economy: {bowler_stats['economy'].mean():.2f}")

print("\nSample bowler stats:")
print(bowler_stats.head(10))

print("\n" + "="*60)
print("DATA QUALITY VERIFICATION - BOWLERS")
print("="*60)

duplicates_bowler = bowler_stats.groupby(['match_id', 'player']).size()
print(f"Duplicate player-match entries: {(duplicates_bowler > 1).sum()}")

print(f"\nEconomy rate stats:")
print(bowler_stats['economy'].describe())
print(f"Economy rates > 20: {(bowler_stats['economy'] > 20).sum()}")



STAGE 5: BOWLER MATCH-LEVEL AGGREGATION

Bowlers with 0 balls bowled: 1

Bowler stats shape: (13865, 15)
Unique bowlers: 550
Average wickets per match: 1.00
Average economy: 8.55

Sample bowler stats:
   match_id       date      player                         team  \
0    335982 2008-04-18   AA Noffke  Royal Challengers Bangalore   
1    335982 2008-04-18  AB Agarkar        Kolkata Knight Riders   
2    335982 2008-04-18    AB Dinda        Kolkata Knight Riders   
3    335982 2008-04-18    CL White  Royal Challengers Bangalore   
4    335982 2008-04-18    I Sharma        Kolkata Knight Riders   
5    335982 2008-04-18   JH Kallis  Royal Challengers Bangalore   
6    335982 2008-04-18   LR Shukla        Kolkata Knight Riders   
7    335982 2008-04-18     P Kumar  Royal Challengers Bangalore   
8    335982 2008-04-18    SB Joshi  Royal Challengers Bangalore   
9    335982 2008-04-18  SC Ganguly        Kolkata Knight Riders   

                      opponent                  venue       

In [6]:
# ============================================================
# STAGE 6: CREATE TARGET VARIABLES (NEXT MATCH PERFORMANCE)
# ============================================================
print("\n" + "="*60)
print("STAGE 6: CREATING TARGET VARIABLES")
print("="*60)

batsman_stats = batsman_stats.sort_values(['player', 'date']).reset_index(drop=True)
bowler_stats = bowler_stats.sort_values(['player', 'date']).reset_index(drop=True)

batsman_stats['target_runs'] = batsman_stats.groupby('player')['runs'].shift(-1)
batsman_stats['target_balls_faced'] = batsman_stats.groupby('player')['balls_faced'].shift(-1)
batsman_stats['target_strike_rate'] = batsman_stats.groupby('player')['strike_rate'].shift(-1)

bowler_stats['target_wickets'] = bowler_stats.groupby('player')['wickets'].shift(-1)
bowler_stats['target_runs_conceded'] = bowler_stats.groupby('player')['runs_conceded'].shift(-1)
bowler_stats['target_economy'] = bowler_stats.groupby('player')['economy'].shift(-1)

batsman_stats['next_opponent'] = batsman_stats.groupby('player')['opponent'].shift(-1)
batsman_stats['next_venue'] = batsman_stats.groupby('player')['venue'].shift(-1)

bowler_stats['next_opponent'] = bowler_stats.groupby('player')['opponent'].shift(-1)
bowler_stats['next_venue'] = bowler_stats.groupby('player')['venue'].shift(-1)

print(f"\nBatsman stats with targets: {batsman_stats['target_runs'].notna().sum()} valid rows")
print(f"Bowler stats with targets: {bowler_stats['target_wickets'].notna().sum()} valid rows")



STAGE 6: CREATING TARGET VARIABLES

Batsman stats with targets: 16937 valid rows
Bowler stats with targets: 13315 valid rows


In [7]:
# ============================================================
# STAGE 7: FILTER PLAYERS WITH SUFFICIENT DATA
# ============================================================
print("\n" + "="*60)
print("STAGE 7: FILTERING PLAYERS")
print("="*60)

batsman_match_counts = batsman_stats.groupby('player').size()
active_batsmen = batsman_match_counts[batsman_match_counts >= 20].index
batsman_stats_filtered = batsman_stats[batsman_stats['player'].isin(active_batsmen)].copy()

bowler_match_counts = bowler_stats.groupby('player').size()
active_bowlers = bowler_match_counts[bowler_match_counts >= 20].index
bowler_stats_filtered = bowler_stats[bowler_stats['player'].isin(active_bowlers)].copy()

print(f"\nActive batsmen (>=20 matches): {len(active_batsmen)}")
print(f"Active bowlers (>=20 matches): {len(active_bowlers)}")
print(f"\nFiltered batsman stats: {batsman_stats_filtered.shape}")
print(f"Filtered bowler stats: {bowler_stats_filtered.shape}")




STAGE 7: FILTERING PLAYERS

Active batsmen (>=20 matches): 226
Active bowlers (>=20 matches): 190

Filtered batsman stats: (14805, 20)
Filtered bowler stats: (11542, 20)


In [8]:
# ============================================================
# STAGE 8: SAVE PREPROCESSED DATA
# ============================================================
print("\n" + "="*60)
print("STAGE 8: SAVING PREPROCESSED DATA")
print("="*60)

batsman_stats_filtered.to_csv('batsman_match_stats.csv', index=False)
bowler_stats_filtered.to_csv('bowler_match_stats.csv', index=False)

df.to_csv('cleaned_ball_by_ball.csv', index=False)

print("\n✅ Saved files:")
print("   - batsman_match_stats.csv")
print("   - bowler_match_stats.csv")
print("   - cleaned_ball_by_ball.csv")


STAGE 8: SAVING PREPROCESSED DATA

✅ Saved files:
   - batsman_match_stats.csv
   - bowler_match_stats.csv
   - cleaned_ball_by_ball.csv


In [9]:
# ============================================================
# STAGE 9: SUMMARY STATISTICS
# ============================================================
print("\n" + "="*60)
print("STAGE 9: PREPROCESSING SUMMARY")
print("="*60)

print(f"\n📊 BATSMAN DATASET:")
print(f"   Total records: {len(batsman_stats_filtered):,}")
print(f"   Players: {batsman_stats_filtered['player'].nunique()}")
print(f"   Matches: {batsman_stats_filtered['match_id'].nunique()}")
print(f"   Date range: {batsman_stats_filtered['date'].min()} to {batsman_stats_filtered['date'].max()}")
print(f"   Seasons: {batsman_stats_filtered['season'].unique()}")

print(f"\n📊 BOWLER DATASET:")
print(f"   Total records: {len(bowler_stats_filtered):,}")
print(f"   Players: {bowler_stats_filtered['player'].nunique()}")
print(f"   Matches: {bowler_stats_filtered['match_id'].nunique()}")
print(f"   Date range: {bowler_stats_filtered['date'].min()} to {bowler_stats_filtered['date'].max()}")
print(f"   Seasons: {bowler_stats_filtered['season'].unique()}")

print(f"\n📊 TARGET VARIABLE STATS (BATSMEN - Runs in Next Match):")
print(batsman_stats_filtered['target_runs'].describe())

print(f"\n📊 TARGET VARIABLE STATS (BOWLERS - Wickets in Next Match):")
print(bowler_stats_filtered['target_wickets'].describe())

print(f"\n📊 SAMPLE BATSMAN DATA:")
print(batsman_stats_filtered[['player', 'date', 'team', 'opponent', 'venue', 
                               'runs', 'balls_faced', 'strike_rate', 'target_runs']].head(10))

print(f"\n📊 SAMPLE BOWLER DATA:")
print(bowler_stats_filtered[['player', 'date', 'team', 'opponent', 'venue',
                              'wickets', 'runs_conceded', 'economy', 'target_wickets']].head(10))

print("\n✅ PREPROCESSING COMPLETE!")
print("="*60)



STAGE 9: PREPROCESSING SUMMARY

📊 BATSMAN DATASET:
   Total records: 14,805
   Players: 226
   Matches: 1169
   Date range: 2008-04-18 00:00:00 to 2025-06-03 00:00:00
   Seasons: [2012 2013 2015 2016 2022 2023 2024 2025 '2007/08' 2009 '2009' '2009/10'
 '2011' 2011 2014 2017 2018 '2019' 2019 '2020/21' '2021' 2021]

📊 BOWLER DATASET:
   Total records: 11,542
   Players: 190
   Matches: 1169
   Date range: 2008-04-18 00:00:00 to 2025-06-03 00:00:00
   Seasons: [2012 2013 2015 2016 '2007/08' '2009' 2009 '2009/10' '2011' 2011 2014 2017
 2018 2019 '2019' '2020/21' '2021' 2021 2023 2024 2022 2025]

📊 TARGET VARIABLE STATS (BATSMEN - Runs in Next Match):
count    14579.000000
mean        22.052679
std         22.179981
min          0.000000
25%          5.000000
50%         15.000000
75%         33.000000
max        175.000000
Name: target_runs, dtype: float64

📊 TARGET VARIABLE STATS (BOWLERS - Wickets in Next Match):
count    11352.000000
mean         1.030303
std          1.065537
min     